# 36 · Load: BLS OOH job descriptions and O\*NET OnLine In Demand % → `staging`

**Source cards:** `bls_ooh.md` (explored in nb 15) and `onet_in_demand.md` (nb 16). Two text sources for the job page (`docs/page3-build-plan.md` §4 items 1–3): the one-line description under the title, and "Skills employers ask for".

| Table | One row = | Expected rows |
|---|---|---|
| `stg_bls__ooh_description` | SOC code covered by an OOH profile, with the profile's one-line description | 606 |
| `stg_onet__in_demand_pct` | O\*NET-SOC code × software product O\*NET shows as In Demand, with its % of the job's U.S. postings | 1,302 |

**BLS OOH** (public domain): every SOC code in the file, not only the job page's; entities decoded; *Military Careers* (no SOC code) drops out. The hand-check seed `ref/ref_job_description_override.csv` is re-checked here word for word against the file, and applied in `dim_job` at step 2.

**O\*NET OnLine** (Lightcast postings, U.S., 2025; CC BY 4.0 for O\*NET content, the percentages' terms pending O\*NET's answer): the 254 pages under the 202 named job-page jobs, retrieved 2026-10-16. Codes whose page shows no product have no row. The tie-out against the O\*NET 31.0 In Demand flags (nb 16 §4) runs again before staging.

In [1]:
import re
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from job_text import check_description_overrides, parse_in_demand_page, parse_ooh, read_in_demand_csv
from staging import connect, raw_files, stage

pd.set_option("display.max_columns", 30, "display.max_colwidth", 100, "display.width", 220)
NB = "36-load-ooh-in-demand.ipynb"
conn = connect()

## 1. BLS OOH descriptions

In [2]:
row = raw_files("bls_ooh").iloc[-1]
ooh = parse_ooh(row.path)
print(row.file, "|", ooh["meta"]["last_updated"], "| profiles", len(ooh["profiles"]), "| SOC rows", len(ooh["socs"]))
o = ooh["socs"].merge(ooh["profiles"], on="ooh_code", validate="many_to_one")
assert o.soc_code.str.fullmatch(r"\d\d-\d{4}").all() and o.soc_code.is_unique
assert (o.description.str.len() > 0).all() and not o.description.str.contains(r"<|&\w+;|\xa0").any()
print("entities decoded in", (o.description != o.description_raw.str.strip()).sum(), "rows")

seed = pd.read_csv(PROJECT / "ref" / "ref_job_description_override.csv", dtype=str, keep_default_na=False)
ok = check_description_overrides(seed, ooh)
assert ok.all(), seed[~ok]
assert set(seed.soc_code) <= set(o.soc_code)
print(f"override seed: {len(seed)} rows, every replacement is BLS's own wording; verdicts {seed.verdict.value_counts().to_dict()}")

stg = (o.rename(columns={"n_soc": "n_soc_in_profile"})
        .assign(n_soc_in_profile=lambda d: d.n_soc_in_profile.astype("Int64"),
                publish_date=lambda d: pd.to_datetime(d.publish_date, format="%Y-%m-%d"), source_file=row.file)
        [["soc_code", "ooh_code", "ooh_title", "description", "n_soc_in_profile", "publish_date", "source_file"]]
        .sort_values("soc_code").reset_index(drop=True))
display(stg[stg.soc_code.isin(["13-2011", "33-3031"])])
display(stage(stg, "stg_bls__ooh_description", ["soc_code"], expected_rows=606, source_key="bls_ooh", notebook=NB, conn=conn,
              comment="BLS Occupational Outlook Handbook (2025-35 edition, file updated 2026-08-26): each SOC code's profile and its "
                      "one-line description, entities decoded. Shared profiles: see ref_job_description_override."))

skills/bls_ooh/xml-compilation_2026-10-16.xml | 8/26/2026 | profiles 343 | SOC rows 606
entities decoded in 3 rows
override seed: 63 rows, every replacement is BLS's own wording; verdicts {'keep': 47, 'profile_part': 9, 'bls_type': 7}


,soc_code,ooh_code,ooh_title,description,n_soc_in_profile,publish_date,source_file
48,13-2011,B047,Accountants and Auditors,Accountants and auditors prepare and examine financial records.,1,2026-08-27,skills/bls_ooh/xml-compilation_2026-10-16.xml
346,33-3031,B213,Police and Detectives,Police officers protect lives and property. Detectives and criminal investigators gather facts a...,4,2026-08-27,skills/bls_ooh/xml-compilation_2026-10-16.xml


  stg_bls__ooh_description: 606 rows x 7 cols, key ['soc_code'] unique | local data/interim/staging/stg_bls__ooh_description.parquet | DB: 606


,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_bls__ooh_description,606,7,soc_code,bls_ooh,36-load-ooh-in-demand.ipynb,606,2026-10-16T17:02:01+00:00


## 2. O\*NET OnLine In Demand %

In [3]:
man = raw_files("onet_in_demand")
man["onet_soc_code"] = man.file.str.extract(r"In_Demand_(\d\d-\d{4})-(\d\d)_").agg(".".join, axis=1)
man["retrieved"] = man.file.str.extract(r"_(\d{4}-\d\d-\d\d)\.\w+$")[0]
man["ext"] = man.file.str.rsplit(".", n=1).str[-1]
latest = man[man.retrieved == man.groupby("onet_soc_code").retrieved.transform("max")]
assert latest.groupby("onet_soc_code").ext.apply(lambda e: sorted(e) == ["csv", "html"]).all(), "every code needs its CSV and its page"
files = latest.pivot(index="onet_soc_code", columns="ext", values="path").join(latest.groupby("onet_soc_code").retrieved.first())
print(f"pages: {len(files)} | retrieved: {files.retrieved.value_counts().to_dict()}")

rows, pages = [], []
for code, r in files.iterrows():
    t = read_in_demand_csv(r.csv)
    p = parse_in_demand_page(r.html)
    assert p["page_code"] == code and p["n_shown"] == len(t) and p["window_start"], (code, p, len(t))
    pages.append({"onet_soc_code": code, **p})
    rows.append(t.assign(onet_soc_code=code, retrieved_at=r.retrieved, window_start=p["window_start"], window_end=p["window_end"],
                         source_file=Path(r.csv).relative_to(PROJECT / "data" / "raw").as_posix()))
d = pd.concat(rows, ignore_index=True)
pages = pd.DataFrame(pages)
print("windows:", pages.groupby(["vendor", "geography", "window_start", "window_end"]).size().to_dict())
print(f"rows {len(d):,} | codes with a product {d.onet_soc_code.nunique()} of {len(pages)}")

pages: 254 | retrieved: {'2026-10-16': 254}


windows: {('Lightcast', 'US nationwide', 'January 1, 2025', 'December 31, 2025'): 254}
rows 1,302 | codes with a product 204 of 254


**Checks:** whole percentages between 1 and 100, sorted high to low within each code, hot flag 'Yes' or blank, and the product set equal to the O\*NET 31.0 In Demand flags for every page (both ways).

In [4]:
d["pct_of_job_ads"] = pd.to_numeric(d.Percentage, errors="raise")
assert (d.pct_of_job_ads % 1 == 0).all() and d.pct_of_job_ads.between(1, 100).all()
assert d.groupby("onet_soc_code").pct_of_job_ads.apply(lambda s: s.is_monotonic_decreasing).all()
assert d["Hot Technology"].isin(["Yes", ""]).all()   # O*NET leaves it blank when the product isn't a Hot Technology
if conn is not None:
    flags = pd.DataFrame(conn.execute("""select onet_soc_code, product from staging.stg_onet__software
                                         where onet_soc_code = any(%s) group by 1, 2 having bool_or(is_in_demand)""",
                                      [pages.onet_soc_code.tolist()]).fetchall(), columns=["onet_soc_code", "product"])
    a = set(map(tuple, flags.values))
    b = set(zip(d.onet_soc_code, d["Software Skill"]))
    print(f"tie-out vs O*NET 31.0 In Demand flags: {len(a)} flagged pairs, {len(b)} live pairs, only 31.0 {len(a - b)}, only live {len(b - a)}")
    assert a == b
else:
    print("tie-out skipped: no database connection (nb 16 §4 holds it)")

stg2 = (d.rename(columns={"Software Skill": "product"})
         .assign(soc_code=lambda x: x.onet_soc_code.str[:7],
                 pct_of_job_ads=lambda x: x.pct_of_job_ads.astype("Int64"),
                 is_hot_technology=lambda x: x["Hot Technology"].eq("Yes"),
                 csv_order=lambda x: x.csv_order.astype("Int64"),
                 window_start=lambda x: pd.to_datetime(x.window_start, format="%B %d, %Y"),
                 window_end=lambda x: pd.to_datetime(x.window_end, format="%B %d, %Y"),
                 retrieved_at=lambda x: pd.to_datetime(x.retrieved_at, format="%Y-%m-%d"))
         [["onet_soc_code", "soc_code", "product", "pct_of_job_ads", "is_hot_technology", "csv_order",
           "window_start", "window_end", "retrieved_at", "source_file"]]
         .sort_values(["onet_soc_code", "csv_order"]).reset_index(drop=True))
display(stg2[stg2.onet_soc_code == "13-2011.00"])
display(stage(stg2, "stg_onet__in_demand_pct", ["onet_soc_code", "product"], expected_rows=1_302,
              source_key="onet_in_demand", notebook=NB, conn=conn,
              comment="O*NET OnLine In Demand Software Skills: % of the job's unique U.S. postings mentioning each product "
                      "(Lightcast, 2025-01-01 to 2025-12-31), for the 254 O*NET-SOC codes under the 202 named job-page jobs, "
                      "retrieved 2026-10-16. Codes whose page shows no product have no row. Reuse terms for the % pending O*NET."))
if conn is not None:
    conn.close()

tie-out vs O*NET 31.0 In Demand flags: 1302 flagged pairs, 1302 live pairs, only 31.0 0, only live 0


,onet_soc_code,soc_code,product,pct_of_job_ads,is_hot_technology,csv_order,window_start,window_end,retrieved_at,source_file
267,13-2011.00,13-2011,Microsoft Excel,28,True,1,2025-01-01,2025-12-31,2026-10-16,skills/onet_in_demand/In_Demand_13-2011-00_2026-10-16.csv
268,13-2011.00,13-2011,Microsoft Office software,17,True,2,2025-01-01,2025-12-31,2026-10-16,skills/onet_in_demand/In_Demand_13-2011-00_2026-10-16.csv
269,13-2011.00,13-2011,Intuit QuickBooks,8,True,3,2025-01-01,2025-12-31,2026-10-16,skills/onet_in_demand/In_Demand_13-2011-00_2026-10-16.csv
270,13-2011.00,13-2011,SAP software,6,True,4,2025-01-01,2025-12-31,2026-10-16,skills/onet_in_demand/In_Demand_13-2011-00_2026-10-16.csv
271,13-2011.00,13-2011,Microsoft PowerPoint,6,True,5,2025-01-01,2025-12-31,2026-10-16,skills/onet_in_demand/In_Demand_13-2011-00_2026-10-16.csv
272,13-2011.00,13-2011,Microsoft Outlook,6,True,6,2025-01-01,2025-12-31,2026-10-16,skills/onet_in_demand/In_Demand_13-2011-00_2026-10-16.csv


  stg_onet__in_demand_pct: 1,302 rows x 10 cols, key ['onet_soc_code', 'product'] unique | local data/interim/staging/stg_onet__in_demand_pct.parquet | DB: 1302


,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_onet__in_demand_pct,1302,10,onet_soc_code + product,onet_in_demand,36-load-ooh-in-demand.ipynb,1302,2026-10-16T17:02:02+00:00
